# Лекција 04 - Образац за коришћење алата

У овој лекцији ћете научити образац за дизајн **коришћења алата** за AI агенте користећи Microsoft Agent Framework (Python). Обрађујемо:

- Дефинисање функција алата са `@tool` декоратером и типизираним параметрима
- Обезбеђивање шема алата како би модел знао шта сваки алат ради
- Контролу извођења алата помоћу `approval_mode`
- Враћање **структурираног излаза** преко Pydantic модела и `response_format`

Сценарио је **агент за резервацију путовања** који може да претражује дестинације, проверава доступност и преузима информације о летовима.


## Подешавање


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Дефинисање алата помоћу @tool декоратора

`@tool` декоратор претвара обичну Python функцију у алат који агент може да позове.
Кључне тачке:

- **Докстринг** постаје опис алата који модел види.
- **Типске анотације** (укључујући `Annotated` са описима) дефинишу шему алата.
- `approval_mode` контролише да ли корисник мора да одобри сваки позив пре него што се изврши.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Креирање агента са више алата

Проследите сва три алата клијенту како би модел могао да позове оне које су му потребне за одговор на питање корисника.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Структурирани излаз са алаткама

Постављањем `response_format` на Пидантички модел, агент је приморан да врати јасно типизиран JSON објекат уместо текста у слободном облику. Ово је корисно када даљи код треба да програмски користи резултат.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## Обрасци одобрења алата

Параметар `approval_mode` на `@tool` контролише да ли позиви алата захтевају људско одобрење пре извршавања:

| Режим | Понашање |
|---|---|
| `"never_require"` | Алат се аутоматски покреће — није потребна потврда корисника. |
| `"always_require"` | Сваком позиву мора претходити одобрење корисника пре извршавања. |

Користите `"always_require"` за алате који имају споредне ефекте (нпр. резервацију лета, наплату кредитне картице) како би човек остао у току.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Резиме

У овој лекцији сте научили како да:

1. **Дефинишете алате** користећи `@tool` декоратор са типизираним параметрима и документацијом која служи као шема алата.
2. **Комбиновати више алата** тако да агент може да их позива у низу како би одговорио на сложена питања.
3. **Вратити структуриран излаз** прослеђивањем Pydantic модела као `response_format`.
4. **Контролисати одобрење алата** помоћу `approval_mode` да бисте укључили човека у процес за осетљиве операције.

Ови обрасци формирају темељ за изградњу поузданих агената спремних за продукцију који могу безбедно да комуницирају са спољним системима.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Одрицање од одговорности**:  
Овaј документ је преведен коришћењем услуге за аутоматски превод [Co-op Translator](https://github.com/Azure/co-op-translator). Иако тежимо прецизности, молимо вас да имате у виду да аутоматски преводи могу да садрже грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати ауторитетним извором. За критичне информације препорука је коришћење професионалног људског превода. Нисмо одговорни за било каква непоразумевања или погрешне тумачења настала употребом овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
